<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/09_Blocking_Improvement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 - Blocking Improvement

This notebook measures how much the **candidate-generation ceiling** improves
when two complementary blockers are added to the frozen three.

| Blocker | Idea | Targets |
| ------- | ---- | ------- |
| `exact_name` | `(country, name_core)` | identical names |
| `name_token` | name tokens (frequency filtered) | word-order / extra words |
| `address_token` | address tokens (frequency filtered) | address-only agreement |
| **`char3`** | shared 3-grams, >= 2 shared | typos, spelling noise |
| **`phonetic`** | country-keyed consonant skeleton of the full name and of the first meaningful token | transliteration / cross-script pairs |

The phonetic key collapses vowels and equivalence classes (g/j, c/k/q, s/z,
v/w) so `green logistics` and its Indic transliteration `grina lajistiks`
share the skeleton `jrnljstks`.

The notebook reuses the package in `code/business_entity_resolution` (cloned
below), so the code is not duplicated.

Expected: the frozen run should reproduce the ~87.24% S2 recall baseline, and
the advanced run should exceed it while keeping candidate volume tractable.


## Load the pipeline package

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/kartik815/Amazon-ML-Challenge-2026.git"
CODE_ROOT = "/content/amlc"

if not os.path.isdir(os.path.join(CODE_ROOT, "code")):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_ROOT], check=False)

PIPELINE_ROOT = os.path.join(CODE_ROOT, "code", "business_entity_resolution")
if PIPELINE_ROOT not in sys.path:
    sys.path.insert(0, PIPELINE_ROOT)

print("pipeline root:", PIPELINE_ROOT)


## Paths

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"
CLEANED_DATA_ROOT = os.path.join(DRIVE_ROOT, "03_Experiments", "Cleaned_Data")
TRAIN_ROOT = os.path.join(
    DRIVE_ROOT, "01_Dataset", "6ab10eb3b23ba_student_resource",
    "student_resource", "dataset", "train"
)

S1_PATH = os.path.join(CLEANED_DATA_ROOT, "train_s1_cleaned.tsv")
SOURCE_PATHS = {
    "S2": os.path.join(CLEANED_DATA_ROOT, "train_s2_cleaned.tsv"),
    "S3": os.path.join(CLEANED_DATA_ROOT, "train_s3_cleaned.tsv"),
}
GT_PATH = os.path.join(TRAIN_ROOT, "train_ground_truth.tsv")

for path in [S1_PATH, GT_PATH, *SOURCE_PATHS.values()]:
    print(("OK   " if os.path.exists(path) else "MISS "), path)


## Baseline: frozen blockers

In [ ]:
from src.config import FROZEN_BLOCKERS, ADVANCED_BLOCKERS
from src import evaluate_blocking

print("=== FROZEN blockers ===")
frozen_result = evaluate_blocking.evaluate(
    S1_PATH, SOURCE_PATHS, GT_PATH, FROZEN_BLOCKERS
)
evaluate_blocking.report(frozen_result)


## Improved: advanced blockers

In [ ]:
print("=== ADVANCED blockers ===")
advanced_result = evaluate_blocking.evaluate(
    S1_PATH, SOURCE_PATHS, GT_PATH, ADVANCED_BLOCKERS
)
evaluate_blocking.report(advanced_result)


## Compare and save

In [ ]:
import json
import os

import pandas as pd

comparison = pd.DataFrame([
    {
        "blocker_set": "frozen",
        "recall": frozen_result["recall_union"],
        "captured": frozen_result["captured_union"],
        "cand_mean": frozen_result["candidate_stats"]["mean"],
        "cand_p99": frozen_result["candidate_stats"]["p99"],
        "cand_zero_s1": frozen_result["candidate_stats"]["zero_s1"],
        "hard_misses": frozen_result["s1_hard_misses"],
    },
    {
        "blocker_set": "advanced",
        "recall": advanced_result["recall_union"],
        "captured": advanced_result["captured_union"],
        "cand_mean": advanced_result["candidate_stats"]["mean"],
        "cand_p99": advanced_result["candidate_stats"]["p99"],
        "cand_zero_s1": advanced_result["candidate_stats"]["zero_s1"],
        "hard_misses": advanced_result["s1_hard_misses"],
    },
])

print(comparison.to_string(index=False))
print()
print("recall gain        :", f"+{advanced_result['recall_union'] - frozen_result['recall_union']:.4%}")
print("matches recovered  :", f"+{advanced_result['incremental_captured']:,}")

out_dir = os.path.join(DRIVE_ROOT, "03_Experiments", "Model")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "blocking_eval.json")
with open(out_path, "w", encoding="utf-8") as handle:
    json.dump({"frozen": frozen_result, "advanced": advanced_result}, handle, indent=2)
print("wrote", out_path)

print(
    "\nTo train and predict with the improved blockers, run the package with:\n"
    "  python run_pipeline.py all --data-dir dataset --output-dir output --advanced-blocking"
)
